# Joint all-slides data prep

Reproduces `notebooks/application/01_data_prep.ipynb` on all 6 benchmark slides at once
(`crc_120, 210, 221, 231, 232, 242` — the set used by `scripts/train_parallel.py`;
`crc_110` is excluded there and here).

Differences from the single-slide notebook, all forced by joining slides:

| | single slide | joint |
|---|---|---|
| HVGs | 2000, per slide | union: in the top 3000 of >= 3 slides (`batch_key='sid'`) |
| spatial graph | one graph | one graph **per slide** via `library_key='sid'` (block-diagonal) |
| `batch_key` | `sid` | `sid` (unchanged) |
| Hotspot fit | all CRC cells | 33k CRC cells per slide (~198k), modules then projected to all ~1.3M |
| counterfactuals | all control cells | 20k control cells per cell type, stratified by slide x REF/TVA |
| splits | 10% test holdout | none (90/10 train/val) - no held-out prediction is evaluated here |
| `microenvironment` for non-CRC | slide-prefixed `typ` (`210_REF`) | `typ_clean` (`REF`/`TVA`), so `Control` is one category across slides |

`spatial_x` and `obsp` are **not** written to disk — they are deterministic functions of
`obsm['spatial']` + `layers['counts']` and are rebuilt in 02 by one `build_spatial()` call.

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import time

sys.path.append('../../scripts')
sys.path.append('../application')

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from tqdm import tqdm

import cellina
from cellina import Cellina as CellinaModel  # renamed upstream before v0.99.1
from utils import set_seed

import joint
from joint import (BATCH_KEY, CELLTYPES, DOMAINS_KEY, LABELS_KEY, SLIDES,
                   build_spatial, cap_per_group, load_joint, lognorm_mean,
                   project_module_scores, slim)

print('cellina', cellina.__version__)

In [ ]:
set_seed(0)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 14
plt.rcParams['figure.dpi'] = 100

OUT = 'output'
FIG = '../../figures/application_all_slides'
os.makedirs(OUT, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# Hotspot: flat cap per slide -> equal contribution per slide, ~198k cells over 6 slides.
# All 6 slides have >72k CRC-region cells, so every slide reaches the cap.
HOTSPOT_CAP_PER_SLIDE = 33_000
HOTSPOT_N_NEIGHBORS = 30
# From sweep.py over top_k x min_genes (output/sweep/sweep_summary.csv): min_genes is the
# only knob that matters. The NF-kB module is 27 genes, so any floor above ~27 either drops
# it or merges it into a 150-629 gene module, diluting PROGENy NFkB activity from 12.3
# (padj 1e-30) to 2.0 (padj 0.21). top_k=750 over 1000: identical NF-kB gene list, but
# 96/750 genes unassigned instead of 260/1000.
HOTSPOT_TOP_K = 750
HOTSPOT_MIN_GENES = 25

# Counterfactuals: ~20k control cells per cell type over 6 slides x {REF, TVA} = 12 groups.
CTRL_CAP_PER_GROUP = 1_667
# Observed CRC-region cells kept per slide x module, for the counterfactual UMAP in 02.
REAL_CAP_PER_GROUP = 1_000
# Only this cell type keeps full counterfactual matrices; the rest keep mean vectors only.
CF_FULL_CELLTYPE = 'Fibroblast'

MODEL_DIR = f'{OUT}/model_final'
CKPT_DIR = 'checkpoints'
N_JOBS = int(os.environ.get('N_JOBS', 24))

SLIDES_USED = list(SLIDES)
# HVG panel: keep a gene if it is in the top N_TOP_GENES of at least MIN_SLIDES slides.
# The panel size is then an outcome of cross-slide reproducibility, not a fixed number.
N_TOP_GENES = 3000
MIN_SLIDES = 3
MAX_EPOCHS = 100

# RESUME=1 in the env reuses an already-trained MODEL_DIR instead of retraining, so a
# restart after a crash costs only load+graph+latents rather than the whole run.
RESUME = os.environ.get('RESUME', '0') == '1'

## Load and join

In [ ]:
t0 = time.time()
adata = load_joint(SLIDES_USED, n_top_genes=N_TOP_GENES, min_slides=MIN_SLIDES)
print(f'\n{adata.n_obs:,} cells x {adata.n_vars:,} genes in {time.time() - t0:.0f}s')
adata

In [ ]:
comp = pd.crosstab(adata.obs['sid'], adata.obs[DOMAINS_KEY])
comp['total'] = comp.sum(axis=1)
display(comp)

print('slide -> patient:')
print(adata.obs[['sid', 'pid']].drop_duplicates().sort_values('sid').to_string(index=False))
print(f"\n{adata.obs['pid'].nunique()} patients across {adata.obs['sid'].nunique()} slides")
display(adata.obs[LABELS_KEY].value_counts())

## Spatial graph (per slide) and neighbourhood features

In [ ]:
t0 = time.time()
build_spatial(adata)
C = adata.obsp['spatial_connectivities']
print(f'built in {time.time() - t0:.0f}s')
print(f'edges/cell     {C.nnz / adata.n_obs:.1f}')
print(f'spatial_x nnz/cell {adata.obsm["spatial_x"].nnz / adata.n_obs:.1f}')

# no edge may cross a slide boundary
coo = C.tocoo()
sid = adata.obs[BATCH_KEY].to_numpy()
n_cross = int(((sid[coo.row] != sid[coo.col]) & (coo.data != 0)).sum())
assert n_cross == 0, f'{n_cross} cross-slide edges'
print('cross-slide edges: 0')

## Train

No holdout: unlike the single-slide notebook this analysis never evaluates held-out
prediction -- every cell is embedded with the trained model and fed to Hotspot -- so a test
split would only remove ~237k cells from training for nothing. 90/10 train/val, scvi's
default split.

In [ ]:
from scvi.train._callbacks import EarlyStopping, SaveCheckpoint

model_args = {
    'adata': adata,
    'n_latent': 64,
    'n_layers': 3,
    'use_observed_lib_size': True,
    'condition_on_intrinsic': False,
    'gene_likelihood': 'nb',
    'classifier_lambda': 1.,
    'discriminator_lambda': 1.,
}
train_args = {
    'max_epochs': MAX_EPOCHS,
    'batch_size': 4096,
    'check_val_every_n_epoch': 1,
    'early_stopping': True,
    'devices': [0],
    'train_size': 0.9,
    'enable_checkpointing': True,
    'callbacks': [
        SaveCheckpoint(monitor='vae_loss_validation', dirpath=CKPT_DIR, load_best_on_end=True),
        EarlyStopping(monitor='vae_loss_validation', patience=5, mode='min'),
    ],
}
plan_kwargs = {'lr': 1e-3, 'normalize_losses': True}

In [ ]:
CellinaModel.setup_anndata(adata,
                           batch_key=BATCH_KEY,
                           labels_key=LABELS_KEY,
                           domains_key=DOMAINS_KEY,
                           spatial_obsm_key='spatial_x',
                           layer='counts')

In [ ]:
t0 = time.time()
if RESUME and os.path.isdir(MODEL_DIR):
    model = CellinaModel.load(MODEL_DIR, adata=adata)
    print('RESUME: loaded', MODEL_DIR)
else:
    model = CellinaModel(**model_args)
    model.train(**train_args, plan_kwargs=plan_kwargs)
    print(f'trained in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Explicit save of the best model (load_best_on_end already restored it),
# so 02 does not have to guess which checkpoint directory is newest.
model.save(MODEL_DIR, overwrite=True)
print('saved', MODEL_DIR)

## Latents

In [ ]:
# give_mean=True is essential, not cosmetic. cellina's get_latent_representation defaults to
# give_mean=False, i.e. it SAMPLES from the posterior (scvi's own default is True). Hotspot
# builds its KNN graph on cellina_spatial, so with a sampled latent the modules differ every
# run: one draw split the NF-kB genes into a 27-gene module, another merged all 26 of them
# into the 383-gene EMT module. The posterior mean (qsm) makes the analysis deterministic.
adata.obsm['cellina_basal'] = model.get_latent_representation(
    adata=adata, latent_key='z', give_mean=True, batch_size=4096)
adata.obsm['cellina_spatial'] = model.get_latent_representation(
    adata=adata, latent_key='s', give_mean=True, batch_size=4096)
print(adata.obsm['cellina_basal'].shape, adata.obsm['cellina_spatial'].shape)

## Hotspot modules

Fitted on a slide-balanced subsample, then **projected** onto every CRC-region cell.
Hotspot's KNN graph lives in `cellina_spatial` latent space, not physical space, so the
projection uses the identical graph-smoothed scoring function on a larger graph.

In [ ]:
import hotspot

crc_pos = np.flatnonzero(adata.obs[DOMAINS_KEY].astype(str).str.contains('CRC').to_numpy())
print(f'{len(crc_pos):,} CRC-region cells')

sub_rel = cap_per_group(adata.obs.iloc[crc_pos], ['sid'], HOTSPOT_CAP_PER_SLIDE, seed=0)
fit_pos = crc_pos[sub_rel]
adata_fit = slim(adata, fit_pos,
                 obs_cols=['sid', 'pid', 'typ', DOMAINS_KEY, LABELS_KEY, 'nCount_RNA'],
                 obsm_keys=['cellina_spatial'])
print(f'Hotspot fit on {adata_fit.n_obs:,} cells')
display(adata_fit.obs['sid'].value_counts())

In [ ]:
set_seed(0)  # reproducible Hotspot KNN + module detection

hs = hotspot.Hotspot(
    adata_fit,
    layer_key='counts',
    model='danb',
    latent_obsm_key='cellina_spatial',
    umi_counts_obs_key='nCount_RNA',
)
hs.create_knn_graph(weighted_graph=False, n_neighbors=HOTSPOT_N_NEIGHBORS)

In [ ]:
t0 = time.time()
hs_results = hs.compute_autocorrelations(jobs=N_JOBS)
print(f'autocorrelations in {(time.time() - t0) / 60:.1f} min')
hs_genes = hs_results.loc[hs_results.FDR < 0.05].head(HOTSPOT_TOP_K).index
print(f'{len(hs_genes)} genes with FDR<0.05 (capped at {HOTSPOT_TOP_K})')

In [ ]:
t0 = time.time()
lcz = hs.compute_local_correlations(hs_genes, jobs=N_JOBS)
print(f'local correlations in {(time.time() - t0) / 60:.1f} min')

In [ ]:
modules = hs.create_modules(min_gene_threshold=HOTSPOT_MIN_GENES, core_only=True, fdr_threshold=0.05)
display(modules.value_counts().sort_index())

In [ ]:
# Small CSVs instead of a multi-GB pickle: 02 only reads hs.results[['C']] and hs.modules.
hs.results.to_csv(f'{OUT}/hotspot_results.csv')
hs.modules.rename('Module').to_csv(f'{OUT}/hotspot_modules.csv')
lcz.to_csv(f'{OUT}/hotspot_lcz.csv')
print('saved hotspot tables')

### Project module scores onto every CRC-region cell

In [ ]:
adata_crc = slim(adata, crc_pos,
                 obs_cols=['sid', 'pid', 'typ', DOMAINS_KEY, LABELS_KEY, 'nCount_RNA'],
                 obsm_keys=['cellina_spatial'])

t0 = time.time()
module_scores = project_module_scores(adata_crc, hs.modules,
                                     n_neighbors=HOTSPOT_N_NEIGHBORS)
print(f'scored {module_scores.shape[0]:,} cells x {module_scores.shape[1]} modules '
      f'in {(time.time() - t0) / 60:.1f} min')
module_scores.head()

In [ ]:
top_module = module_scores.idxmax(axis=1)

# non-CRC cells keep typ_clean (REF/TVA) so that 'Control' is one category across slides
micro = adata.obs[DOMAINS_KEY].astype(str).copy()
micro.iloc[crc_pos] = [f'CRC{int(m)}' for m in top_module.reindex(adata.obs_names[crc_pos])]
adata.obs['microenvironment'] = pd.Categorical(micro)

MODULES = sorted([m for m in adata.obs['microenvironment'].cat.categories if str(m).startswith('CRC')],
                 key=lambda s: int(s[3:]))
print('modules:', MODULES)
display(adata.obs['microenvironment'].value_counts())

assert adata.obs['microenvironment'].isna().sum() == 0
assert adata.obs['microenvironment'].iloc[crc_pos].astype(str).str.startswith('CRC').all()

### Is a module just one patient?

The single-slide run structurally cannot answer this. A module whose cells are ~all from
one slide is a patient effect, not a shared programme.

In [ ]:
mod_by_slide = pd.crosstab(adata.obs['microenvironment'], adata.obs['sid']).loc[MODULES]
mod_frac = mod_by_slide.div(mod_by_slide.sum(axis=1), axis=0)

mod_by_slide.to_csv(f'{OUT}/module_slide_composition.csv')
display(mod_by_slide)
print('\nrow-fraction per slide (%):')
display((mod_frac * 100).round(1))
print('\nmax single-slide share per module (%):')
display((mod_frac.max(axis=1) * 100).round(1))

In [ ]:
# crc_120 has pathologist-called CRC subregions -> free external check on that slide
is_120 = (adata.obs['sid'].astype(str) == '120').to_numpy()
if is_120.any():
    display(pd.crosstab(adata.obs.loc[is_120, 'microenvironment'],
                        adata.obs.loc[is_120, 'typ']))

## Counterfactuals

Edge-swapping counterfactuals (`get_counterfactual_expression`) for control cells of each
cell type, conditioned on the global CRC neighbourhood and on each module. Control cells
are capped at ~20k per cell type; every quantity 02 derives from these matrices is a column
mean, which converges long before that.

In [ ]:
counts = adata.layers['counts']
is_crc = np.zeros(adata.n_obs, dtype=bool)
is_crc[crc_pos] = True
target_pos = {'global': crc_pos}
for m in MODULES:
    target_pos[m] = np.flatnonzero((adata.obs['microenvironment'].astype(str) == m).to_numpy())
print({k: len(v) for k, v in target_pos.items()})

In [ ]:
means = {'genes': adata.var_names.to_numpy().astype(str)}

for ct in tqdm(CELLTYPES, desc='cell types'):
    is_ct = (adata.obs[LABELS_KEY].astype(str) == ct).to_numpy()
    ctrl_pool = np.flatnonzero(~is_crc & is_ct)
    if len(ctrl_pool) == 0:
        print(f'{ct}: no control cells, skipping')
        continue
    ctrl_pos = ctrl_pool[cap_per_group(adata.obs.iloc[ctrl_pool], ['sid', DOMAINS_KEY],
                                       CTRL_CAP_PER_GROUP, seed=0)]
    print(f'\n{ct}: {len(ctrl_pool):,} control cells -> {len(ctrl_pos):,} sampled')

    recon_ctrl = model.get_normalized_expression(adata=adata, indices=ctrl_pos,
                                                 batch_size=4096, library_size=1e4)
    cf = {}
    for name, tgt in target_pos.items():
        cf[name] = model.get_counterfactual_expression(
            adata=adata, indices=ctrl_pos, neighbour_indices=tgt,
            batch_size=4096, seed=0, library_size=1e4).astype(np.float32)

    # mean vectors for every cell type (all that the systematic clustermap would need)
    means[f'{ct}|ctrl|recon'] = np.log1p(recon_ctrl).mean(0).astype(np.float32)
    means[f'{ct}|ctrl|obs'] = lognorm_mean(counts, ctrl_pos).astype(np.float32)
    for name, arr in cf.items():
        means[f'{ct}|{name}|cf'] = np.log1p(arr).mean(0).astype(np.float32)
    for m in MODULES:
        obs_pos = np.flatnonzero(is_ct & (adata.obs['microenvironment'].astype(str) == m).to_numpy())
        if len(obs_pos):
            means[f'{ct}|{m}|obs'] = lognorm_mean(counts, obs_pos).astype(np.float32)

    if ct != CF_FULL_CELLTYPE:
        continue

    # full arrays, only for the cell type 02 actually plots
    real_pool = np.flatnonzero(is_crc & is_ct)
    real_pos = real_pool[cap_per_group(adata.obs.iloc[real_pool], ['sid', 'microenvironment'],
                                       REAL_CAP_PER_GROUP, seed=0)]
    recon_real = model.get_normalized_expression(adata=adata, indices=real_pos,
                                                 batch_size=4096, library_size=1e4)
    keep = np.concatenate([ctrl_pos, real_pos])
    sub = ad.AnnData(X=counts[keep].copy(), obs=adata.obs.iloc[keep].copy(),
                     var=pd.DataFrame(index=adata.var_names.copy()))
    sub.layers['counts'] = sub.X.copy()
    sub.obs['arm'] = np.r_[np.repeat('control', len(ctrl_pos)), np.repeat('real', len(real_pos))]
    sub.obsm['recon_x'] = np.vstack([recon_ctrl, recon_real]).astype(np.float32)
    for name, arr in cf.items():
        sub.uns[f'counterfactual_x_{name}'] = arr
    sub.uns['n_control'] = len(ctrl_pos)
    sub.uns['modules'] = list(MODULES)
    sub.write_h5ad(f'{OUT}/cf_{ct}.h5ad')
    print(f'  wrote {OUT}/cf_{ct}.h5ad  ({sub.n_obs:,} cells, {len(cf)} cf matrices)')

np.savez_compressed(f'{OUT}/cf_means.npz', **means)
print(f'\nwrote {OUT}/cf_means.npz  ({len(means) - 1} mean vectors)')

## Save

`spatial_x` and `obsp` are dropped — 02 rebuilds them with one `build_spatial()` call,
which is minutes of compute against ~16GB of disk.

In [ ]:
del adata.obsm['spatial_x']
del adata.obsp['spatial_connectivities']

adata.write_h5ad(f'{OUT}/adata_joint.h5ad')
print('wrote', f'{OUT}/adata_joint.h5ad')
print(f'{adata.n_obs:,} cells x {adata.n_vars:,} genes')
adata